# Counsel Agents

This is a multi-agent AI system that analyses a user's legal case from every angle using five specialist AI agents running in parallel debate:

- Plaintiff's Counsel (Llama 3.3 70B via HuggingFace): builds the strongest argument in favour of the user's position
- Defense Counsel (Llama 3.3 70B via HuggingFace): mounts the toughest counter-argument to stress-test weaknesses
- Expert Witness (Claude Sonnet via OpenRouter): provides objective analysis of applicable statutes, precedents, and burden of proof
- Judge (OpenAI o3 via OpenRouter): scores both sides across five legal criteria and identifies vulnerabilities
- Legal Strategist (GPT-4o via OpenRouter): synthesises all outputs into an actionable case preparation memo
The pipeline streams results step-by-step through a styled Gradio UI, supporting 11 legal areas (Contract, Employment, IP, Criminal, Family Law, and more). Built on a unified LLMAdapter that handles provider differences including models that don't support temperature or JSON mode.

In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, field
from typing import Dict, Generator, List, Optional, Tuple

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

huggingface_token  = os.getenv("HUGGING_FACE_TOKEN")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if huggingface_token:
    print(f"HuggingFace Token exists and begins with '{huggingface_token[:8]}'")
else:
    print("HuggingFace Token not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins with '{openrouter_api_key[:7]}'")
else:
    print("OpenRouter API Key not set")

#### CONFIGURATION DATACLASS & CLIENT FACTORY

In [ ]:
@dataclass
class AgentConfig:
    """Holds all configuration required to talk to an LLM provider."""
    name:                str
    model:               str
    api_key:             str
    base_url:            str
    temperature:         float = 0.7
    supports_json:       bool  = True
    supports_temperature: bool  = True


def make_client(config: AgentConfig) -> OpenAI:
    """Return an OpenAI-compatible client for any provider."""
    if not config.api_key:
        raise RuntimeError(f"Missing API key for agent '{config.name}'.")
    return OpenAI(api_key=config.api_key, base_url=config.base_url)


def extract_text(response) -> str:
    """Extract plain text from an OpenAI-style response object."""
    choices = getattr(response, "choices", None) or (
        response.get("choices") if isinstance(response, dict) else None
    )
    if not choices:
        raise RuntimeError(f"LLM response missing choices: {response!r}")

    choice  = choices[0]
    message = getattr(choice, "message", None) or (
        choice.get("message") if isinstance(choice, dict) else None
    )
    content = None

    if message is not None:
        content = getattr(message, "content", None) or (
            message.get("content") if isinstance(message, dict) else None
        )

    if isinstance(content, list):
        parts: List[str] = []
        for part in content:
            if isinstance(part, dict):
                parts.append(str(
                    part.get("text") or part.get("output_text") or part.get("content", "")
                ))
            else:
                parts.append(str(part))
        content = "".join(parts)

    if content is None:
        content = getattr(choice, "text", None) or (
            choice.get("text") if isinstance(choice, dict) else None
        )

    if content is None:
        raise RuntimeError(f"LLM response missing content: {response!r}")

    return str(content).strip()

#### AGENT CONFIGURATIONS

In [ ]:
# HuggingFace Inference API — OpenAI-compatible serverless endpoint
HF_BASE_URL         = "https://router.huggingface.co/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Plaintiff's Counsel — large open model, strong reasoning, pro-user argument
PLAINTIFF_CONFIG = AgentConfig(
    name        = "Plaintiff's Counsel",
    model       = "meta-llama/Llama-3.3-70B-Instruct",
    api_key     = huggingface_token,
    base_url    = HF_BASE_URL,
    temperature = 0.7,
)

# Defense Counsel — same model family, independent instance, counter-argument
DEFENSE_CONFIG = AgentConfig(
    name        = "Defense Counsel",
    model       = "meta-llama/Llama-3.3-70B-Instruct",
    api_key     = huggingface_token,
    base_url    = HF_BASE_URL,
    temperature = 0.7,
)

# Expert Witness — Claude for precise, objective legal analysis
EXPERT_CONFIG = AgentConfig(
    name          = "Expert Witness",
    model         = "anthropic/claude-sonnet-4-5",
    api_key       = openrouter_api_key,
    base_url      = OPENROUTER_BASE_URL,
    temperature   = 0.2,
    supports_json = True,
)

# Judge — o3 for thorough, balanced legal evaluation
JUDGE_CONFIG = AgentConfig(
    name                 = "Judge",
    model                = "openai/o3",
    api_key              = openrouter_api_key,
    base_url             = OPENROUTER_BASE_URL,
    temperature          = 0.1,
    supports_json        = False,  # o3 reasoning models do not support json_object mode
    supports_temperature = False,  # o3 reasoning models do not support temperature
)

# Legal Strategist
STRATEGIST_CONFIG = AgentConfig(
    name          = "Legal Strategist",
    model         = "openai/gpt-4o",
    api_key       = openrouter_api_key,
    base_url      = OPENROUTER_BASE_URL,
    temperature   = 0.4,
    supports_json = True,
)

#### LLM ADAPTER

In [ ]:
class LLMAdapter:
    """Thin wrapper around the OpenAI SDK; works with any OpenAI-compatible endpoint."""

    def __init__(self, config: AgentConfig):
        self.config = config
        self.client = make_client(config)

    def complete(
        self,
        prompt:     str,
        *,
        system:     Optional[str] = None,
        max_tokens: int  = 600,
        json_mode:  bool = False,
    ) -> str:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        params: dict = dict(
            model      = self.config.model,
            messages   = messages,
            max_tokens = max_tokens,
        )
        if self.config.supports_temperature:
            params["temperature"] = self.config.temperature
        if json_mode and self.config.supports_json:
            params["response_format"] = {"type": "json_object"}

        response = self.client.chat.completions.create(**params)
        return extract_text(response)

#### SPECIALIST AGENTS

In [ ]:
class PlaintiffCounsel:
    """Constructs the strongest legal argument in favour of the user's position."""

    SYSTEM = (
        "You are a seasoned plaintiff's attorney. Your role is to construct "
        "the strongest possible legal argument in FAVOUR of your client's position. "
        "Be strategic, cite relevant legal principles, anticipate weaknesses, "
        "and write in a persuasive but professional legal tone."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def argue(self, case_description: str, legal_area: str, user_position: str) -> str:
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Your client's position: {user_position}\n\n"
            "Build the strongest possible argument for this position. Include:\n"
            "1. Core legal theory\n"
            "2. Key facts that support the claim\n"
            "3. Relevant legal principles or precedents\n"
            "4. Anticipated defence objections and pre-emptive rebuttals\n\n"
            "Be concise but thorough. Max 300 words."
        )
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=500)


class DefenseCounsel:
    """Mounts the strongest counter-argument to stress-test the user's position."""

    SYSTEM = (
        "You are a sharp defense attorney. Your role is to dismantle the opposing "
        "party's legal position. Identify weaknesses, raise procedural issues, "
        "challenge the evidence, and present the strongest counter-arguments possible. "
        "Write in a professional legal tone."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def argue(self, case_description: str, legal_area: str, user_position: str) -> str:
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"The opposing party's position: {user_position}\n\n"
            "Mount the strongest possible counter-argument. Include:\n"
            "1. Primary legal defence or counter-theory\n"
            "2. Factual weaknesses in the opposing case\n"
            "3. Procedural or evidentiary challenges\n"
            "4. Alternative interpretations of the facts\n\n"
            "Be incisive and precise. Max 300 words."
        )
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=500)


class ExpertWitness:
    """Provides objective domain-specific analysis: statutes, precedents, burden of proof."""

    SYSTEM = (
        "You are an independent legal expert and academic. You provide objective "
        "technical analysis of legal matters: relevant statutes, landmark case law, "
        "regulatory frameworks, and doctrinal issues. You do not advocate for either "
        "side — your role is to clarify the legal landscape and identify the key "
        "legal questions a court or regulator would focus on."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def analyse(
        self,
        case_description: str,
        legal_area:       str,
        plaintiff_arg:    str,
        defense_arg:      str,
    ) -> Dict:
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's argument:\n{plaintiff_arg}\n\n"
            f"Defense's argument:\n{defense_arg}\n\n"
            "Provide an expert technical analysis as a JSON object:\n"
            "{\n"
            '  "key_legal_questions": ["question1", "question2"],\n'
            '  "applicable_law": "Summary of relevant statutes or doctrine",\n'
            '  "precedents": "Notable cases or rulings relevant to this matter",\n'
            '  "burden_of_proof": "Who bears the burden and what standard applies",\n'
            '  "critical_risk_factors": ["risk1", "risk2"]\n'
            "}\n"
            "Return only valid JSON."
        )
        raw = self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=600, json_mode=True)
        try:
            clean = raw.strip()
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[-1]
            if clean.endswith("```"):
                clean = clean.rsplit("```", 1)[0]
            clean = clean.strip()
            return json.loads(clean)
        except Exception:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                try:
                    return json.loads(m.group())
                except Exception:
                    pass
            return {
                "key_legal_questions":   [],
                "applicable_law":        raw,
                "precedents":            "",
                "burden_of_proof":       "",
                "critical_risk_factors": [],
            }


class Judge:
    """Evaluates argument strength across legal criteria and surfaces vulnerabilities."""

    RUBRIC = [
        "Legal theory strength",
        "Factual support",
        "Anticipation of counter-arguments",
        "Procedural soundness",
        "Overall persuasiveness",
    ]

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def evaluate(
        self,
        case_description: str,
        legal_area:       str,
        plaintiff_arg:    str,
        defense_arg:      str,
        expert_analysis:  Dict,
    ) -> Dict:
        rubric_text    = "\n".join(f"- {item}" for item in self.RUBRIC)
        expert_summary = json.dumps(expert_analysis, ensure_ascii=False, indent=2)
        prompt = (
            "You are an experienced judge evaluating the quality of legal arguments.\n\n"
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's Counsel argues:\n{plaintiff_arg}\n\n"
            f"Defense Counsel argues:\n{defense_arg}\n\n"
            f"Expert legal analysis:\n{expert_summary}\n\n"
            "Score each side 0-10 on the following criteria:\n"
            f"{rubric_text}\n\n"
            "Return a JSON object:\n"
            "{\n"
            '  "stronger_position": "Plaintiff" or "Defense" or "Balanced",\n'
            '  "judicial_assessment": "Your overall assessment (2-3 sentences)",\n'
            '  "plaintiff_vulnerabilities": ["vuln1", "vuln2"],\n'
            '  "defense_vulnerabilities": ["vuln1", "vuln2"],\n'
            '  "scores": [\n'
            '    {"criterion": "...", "plaintiff": 0-10, "defense": 0-10, "notes": "..."}\n'
            "  ]\n"
            "}\n"
            "Return valid JSON only."
        )
        # o3 needs more tokens: it uses some for internal reasoning before generating output
        raw = self.adapter.complete(prompt, max_tokens=2000, json_mode=True)
        try:
            clean = raw.strip()
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[-1]
            if clean.endswith("```"):
                clean = clean.rsplit("```", 1)[0]
            clean = clean.strip()
            data  = json.loads(clean)
            if "scores" not in data:
                raise ValueError("scores missing")
            return data
        except Exception:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                try:
                    data = json.loads(m.group())
                    if "scores" in data:
                        return data
                except Exception:
                    pass
            return {
                "stronger_position":         "Unknown",
                "judicial_assessment":       raw,
                "plaintiff_vulnerabilities": [],
                "defense_vulnerabilities":   [],
                "scores":                    [],
            }


class LegalStrategist:
    """Synthesises all agent outputs into an actionable case preparation memo."""

    SYSTEM = (
        "You are a senior legal strategist with 30 years of litigation experience. "
        "You synthesise complex legal analysis into clear, actionable advice. "
        "Your goal is to help the user strengthen their case and prepare for "
        "the arguments they will face. Be practical, specific, and candid."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def advise(
        self,
        case_description: str,
        legal_area:       str,
        user_position:    str,
        plaintiff_arg:    str,
        defense_arg:      str,
        expert_analysis:  Dict,
        judge_result:     Dict,
    ) -> str:
        prompt = (
            f"Legal area: {legal_area}\n"
            f"User's position: {user_position}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's argument:\n{plaintiff_arg}\n\n"
            f"Defense's argument:\n{defense_arg}\n\n"
            f"Expert analysis:\n{json.dumps(expert_analysis, indent=2)}\n\n"
            f"Judicial assessment:\n{json.dumps(judge_result, indent=2)}\n\n"
            "Provide a structured case preparation memo:\n\n"
            "STRENGTHS TO LEVERAGE\n"
            "- List 3 strongest points in the user's favour\n\n"
            "VULNERABILITIES TO ADDRESS\n"
            "- List 3 key weaknesses the user must shore up\n\n"
            "EVIDENCE GAPS\n"
            "- What additional evidence or documentation should be gathered?\n\n"
            "RECOMMENDED STRATEGY\n"
            "- Concise recommended litigation or settlement strategy\n\n"
            "IMMEDIATE ACTION ITEMS\n"
            "- 3 to 5 concrete next steps for case preparation\n\n"
            "Max 400 words. Be direct and practical."
        )
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=600)

#### AGENT INSTANTIATION

In [ ]:
plaintiff_counsel = PlaintiffCounsel(LLMAdapter(PLAINTIFF_CONFIG))
defense_counsel   = DefenseCounsel(LLMAdapter(DEFENSE_CONFIG))
expert_witness    = ExpertWitness(LLMAdapter(EXPERT_CONFIG))
judge             = Judge(LLMAdapter(JUDGE_CONFIG))
strategist        = LegalStrategist(LLMAdapter(STRATEGIST_CONFIG))

#### FORMATTING HELPERS

In [ ]:
def _format_expert(expert: Dict) -> str:
    lines = []
    if ql := expert.get("key_legal_questions"):
        lines.append("**Key Legal Questions**")
        lines.extend(f"- {q}" for q in ql)
        lines.append("")
    if al := expert.get("applicable_law"):
        lines.append(f"**Applicable Law**\n{al}\n")
    if pr := expert.get("precedents"):
        lines.append(f"**Relevant Precedents**\n{pr}\n")
    if bp := expert.get("burden_of_proof"):
        lines.append(f"**Burden of Proof**\n{bp}\n")
    if rf := expert.get("critical_risk_factors"):
        lines.append("**Critical Risk Factors**")
        lines.extend(f"- {r}" for r in rf)
    return "\n".join(lines)


def _format_judge(result: Dict) -> str:
    lines = []
    if sp := result.get("stronger_position"):
        lines.append(f"**Stronger Position:** {sp}\n")
    if ja := result.get("judicial_assessment"):
        lines.append(f"**Judicial Assessment**\n{ja}\n")
    if pv := result.get("plaintiff_vulnerabilities"):
        lines.append("**Plaintiff Vulnerabilities**")
        lines.extend(f"- {v}" for v in pv)
        lines.append("")
    if dv := result.get("defense_vulnerabilities"):
        lines.append("**Defense Vulnerabilities**")
        lines.extend(f"- {v}" for v in dv)
    return "\n".join(lines)

#### DEBATE PIPELINE (STREAMING GENERATOR)

In [ ]:
LEGAL_AREAS = [
    "Contract Law",
    "Employment Law",
    "Intellectual Property",
    "Corporate / M&A",
    "Regulatory / Compliance",
    "Personal Injury / Tort",
    "Real Estate / Property",
    "Data Privacy / GDPR",
    "Criminal Law",
    "Family Law",
    "Other",
]


def run_case_analysis(
    case_description: str,
    legal_area:       str,
    user_position:    str,
) -> Generator:
    """Streams results step by step as each agent completes its work."""

    empty = ("", "", "", "", [], "", "")

    if not case_description.strip():
        yield ("Please describe your case to begin.", *empty[1:])
        return
    if not user_position.strip():
        yield ("Please describe your position in this case.", *empty[1:])
        return

    # Step 1 — Plaintiff argues
    yield ("Plaintiff's Counsel is building your argument...", "", "", "", [], "", "")
    p_arg = plaintiff_counsel.argue(case_description, legal_area, user_position)

    # Step 2 — Defense argues
    yield (p_arg, "Defense Counsel is preparing counter-arguments...", "", "", [], "", "")
    d_arg = defense_counsel.argue(case_description, legal_area, user_position)

    # Step 3 — Expert analyses
    yield (p_arg, d_arg, "Expert Witness is analysing applicable law and precedents...", "", [], "", "")
    expert    = expert_witness.analyse(case_description, legal_area, p_arg, d_arg)
    expert_md = _format_expert(expert)

    # Step 4 — Judge evaluates
    yield (p_arg, d_arg, expert_md, "Judge is evaluating argument strength...", [], "", "")
    judge_result = judge.evaluate(case_description, legal_area, p_arg, d_arg, expert)
    score_rows   = [
        [e.get("criterion", ""), e.get("plaintiff", ""), e.get("defense", ""), e.get("notes", "")]
        for e in judge_result.get("scores", [])
    ]
    judge_md = _format_judge(judge_result)

    # Step 5 — Strategist advises
    yield (p_arg, d_arg, expert_md, judge_md, score_rows, "Legal Strategist is preparing your case memo...", "")
    strategy = strategist.advise(
        case_description, legal_area, user_position,
        p_arg, d_arg, expert, judge_result,
    )

    stronger   = judge_result.get("stronger_position", "Unknown")
    summary_md = (
        f"**Overall Assessment:** {judge_md.split(chr(10))[0]}\n\n"
        f"**Stronger Position:** {stronger}"
    )

    yield (p_arg, d_arg, expert_md, judge_md, score_rows, strategy, summary_md)

#### GRADIO UI

In [ ]:
THEME = gr.themes.Default(
    primary_hue   = "slate",
    secondary_hue = "blue",
    neutral_hue   = "gray",
)

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Source+Serif+4:ital,wght@0,300;0,400;1,300&display=swap');

body, .gradio-container {
    background: #0e0e0e;
    color: #e8e2d9;
    font-family: 'Source Serif 4', Georgia, serif;
}
h1, h2, h3 { font-family: 'Playfair Display', Georgia, serif !important; }
.gradio-container { max-width: 1400px !important; }

#plaintiff-panel {
    background: linear-gradient(160deg, #0a1628 0%, #0d1f3c 100%);
    border-left: 4px solid #3b82f6;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #bfdbfe;
    box-shadow: inset 0 0 60px rgba(59,130,246,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#defense-panel {
    background: linear-gradient(160deg, #1a0a0a 0%, #2d0f0f 100%);
    border-left: 4px solid #ef4444;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #fecaca;
    box-shadow: inset 0 0 60px rgba(239,68,68,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#expert-panel {
    background: linear-gradient(160deg, #1a1400 0%, #2d2200 100%);
    border-left: 4px solid #f59e0b;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #fde68a;
    box-shadow: inset 0 0 60px rgba(245,158,11,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#judge-panel {
    background: linear-gradient(160deg, #111111 0%, #1a1a1a 100%);
    border-left: 4px solid #9ca3af;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #e5e7eb;
    box-shadow: inset 0 0 60px rgba(156,163,175,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#strategy-panel {
    background: linear-gradient(160deg, #001a0d 0%, #002d15 100%);
    border-left: 4px solid #10b981;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #a7f3d0;
    box-shadow: inset 0 0 60px rgba(16,185,129,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
.gr-button-primary {
    background: linear-gradient(135deg, #1e3a5f, #1d4ed8) !important;
    border: 1px solid #3b82f6 !important;
    font-family: 'Playfair Display', serif !important;
    letter-spacing: 0.05em !important;
    font-size: 1rem !important;
    padding: 12px 32px !important;
}
.gr-button-primary:hover {
    background: linear-gradient(135deg, #1d4ed8, #2563eb) !important;
    box-shadow: 0 0 20px rgba(59,130,246,0.3) !important;
}
footer { display: none !important; }
"""

with gr.Blocks(
    title      = "Counsel Agents",
    fill_width = True,
    theme      = THEME,
    css        = CUSTOM_CSS,
) as demo:

    gr.Markdown(
        "# Legal Counsel Agents\n"
        "*Five AI legal specialists analyse your case from every angle "
        "so you walk into proceedings fully prepared.*"
    )

    with gr.Row():
        with gr.Column(scale=2):
            case_input = gr.Textbox(
                label       = "Case Description",
                placeholder = (
                    "Describe the facts of your case in as much detail as possible. "
                    "Include key dates, parties involved, actions taken, and any "
                    "relevant documents or communications..."
                ),
                lines = 7,
            )
        with gr.Column(scale=1):
            legal_area_input = gr.Dropdown(
                label   = "Legal Area",
                choices = LEGAL_AREAS,
                value   = "Contract Law",
            )
            position_input = gr.Textbox(
                label       = "Your Position",
                placeholder = (
                    "e.g. 'I am the plaintiff seeking damages for breach of contract' "
                    "or 'We are the defendant arguing the contract was void ab initio'"
                ),
                lines = 4,
            )

    run_button = gr.Button("Analyse My Case", variant="primary", size="lg")

    gr.Markdown("---")
    summary_md = gr.Markdown("")
    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Plaintiff's Counsel")
            plaintiff_out = gr.Markdown(
                "Argument will appear here...", elem_id="plaintiff-panel"
            )
        with gr.Column():
            gr.Markdown("### Defense Counsel")
            defense_out = gr.Markdown(
                "Counter-argument will appear here...", elem_id="defense-panel"
            )

    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Expert Witness Analysis")
            expert_out = gr.Markdown(
                "Legal analysis will appear here...", elem_id="expert-panel"
            )
        with gr.Column():
            gr.Markdown("### Judge's Assessment")
            judge_out = gr.Markdown(
                "Judicial evaluation will appear here...", elem_id="judge-panel"
            )

    gr.Markdown("---")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### Strategic Case Memo")
            strategy_out = gr.Markdown(
                "Your preparation memo will appear here...", elem_id="strategy-panel"
            )
        with gr.Column(scale=1):
            gr.Markdown("### Argument Scores")
            score_table = gr.Dataframe(
                headers     = ["Criterion", "Plaintiff", "Defense", "Notes"],
                datatype    = ["str", "number", "number", "str"],
                interactive = False,
            )

    gr.Markdown(
        "*This tool is for legal research and preparation purposes only. "
        "It does not constitute legal advice. Always consult a qualified attorney.*"
    )

    run_button.click(
        fn      = run_case_analysis,
        inputs  = [case_input, legal_area_input, position_input],
        outputs = [
            plaintiff_out, defense_out, expert_out,
            judge_out, score_table, strategy_out, summary_md,
        ],
        queue = True,
    )

In [ ]:
demo.queue(default_concurrency_limit=4).launch()

In [ ]:
# demo.queue(default_concurrency_limit=4).close()